STEP 1

The Cheat Sheet (Requirement Gathering Principles)

    1) Focus on the problem, not the visuals: Never start a conversation by asking, "What charts do you want to see?". Always ask: "What business problem are you currently facing?".

    2) Define operational expectations: Once the problem is defined, ask what exactly the user expects from the tool and what specific action they will take when they see the data.

    3) Propose an initial solution (Prototyping): Don't build the final report blindly. Sketch an initial layout (even on paper) and make sure it actually solves the business need.

    4) Define the timeframe and frequency: Establish the time context – are decisions made daily, or strategically once a quarter?

    5) Force specific metrics (Prioritization): Limit the number of key indicators to 3-5, instead of agreeing to "show all the data on one screen".

The Interrogation (Stakeholder Questions)
For the Director (Strategic Perspective):

    1. What are the 3 most important strategic goals for the company this year?
    2. By looking at which specific KPIs will you know that we are heading in the right direction and succeeding?
    3. What recent business decision did you have to make blindly because you lacked the right data?
    4. During which meetings will this report be used and presented?

For the Operations Manager (Tactical/Operational Perspective):

    5. What specific problem are you dealing with, and which process generates the most customer complaints?
    6. What exactly do you need to see on your screen every morning to catch a problem before it escalates?
    7. How do you currently track these operations, and what annoys you the most about this process?
    8. When you see a metric deteriorating, what is your first action, and who do you call?

Why is it dangerous to put "all metrics on one screen"?

It is simply unnecessary and causes the data to blend together into visual noise. A dashboard shouldn't just exist for the sake of passively checking numbers. Its primary purpose is to deliver tangible value and drive beneficial business outcomes. Cluttering the screen prevents users from extracting that value.

What is the difference between a Metric and an Actionable Insight?

A metric is just a standalone "number" that we see on a dashboard. An actionable insight, on the other hand, is ready-to-use information deduced from those metrics and KPIs, designed specifically to trigger a further action that drives business development or solves a problem.

STEP 2

## 1. Executive Summary

The goal of the BI solution is to provide two complementary reporting experiences:

- A strategic dashboard for the Executive Director to monitor sales performance, revenue trends, geographical performance, and customer value.
- An operational dashboard for the Operations Manager to identify delivery bottlenecks, delayed orders, and logistics issues requiring action.

The solution uses a PostgreSQL data warehouse with Stage, Core, and Mart layers. Power BI connects to the reporting-ready Mart Star Schema.

## 2. Target Audience & User Personas

**Executive Director (Strategic)**

The Executive Director monitors overall sales performance, revenue trends, geographical results, product categories, and customer segmentation. The dashboard supports identifying high-performing markets, underperforming regions, and valuable customer segments.

**Operations Manager (Tactical)**

The Operations Manager focuses on delivery performance, delayed orders, delivery durations, and geographical bottlenecks. The dashboard helps identify operational issues requiring investigation or corrective action.

## 3. Key Metrics & KPIs

**Sales and Customer Metrics**

- **Total Revenue:** Total sales value including product prices and freight charges, excluding canceled orders.
- **Total Orders:** Number of distinct orders, excluding canceled orders.
- **Average Order Value (AOV):** Total Revenue divided by Total Orders.
- **Total Customers:** Number of distinct purchasing customers within the current filter context.
- **Orders per Customer:** Total Orders divided by Total Customers.
- **Revenue per Customer:** Total Revenue divided by Total Customers.
- **Average Freight:** Average freight charge per order.
- **MoM Growth %:** Month-over-month performance change based on the selected metric.
- **Regional Revenue Share %:** Revenue in the selected geographical context compared with the relevant total within the user's authorized RLS scope.

**Delivery Metrics**

- **Total Delayed Orders:** Number of distinct orders delivered after their estimated delivery date.
- **Avg Delivery Days:** Average time between order purchase and actual customer delivery, calculated once per order.
- **Avg Delay Days:** Average number of days exceeding the estimated delivery date for delayed orders.
- **Delayed Order Rate %:** Percentage of SLA-eligible orders delivered late.
- **% On-Time Delivery:** Percentage of SLA-eligible orders delivered on time.

**Delivery SLA Definition**

An order is:

- **On-Time** when the actual customer delivery date is on or before the estimated delivery date.
- **Delayed** when the actual customer delivery date is after the estimated delivery date.
- **SLA-eligible** when both actual and estimated delivery dates are available.

Delivery duration, delay duration, and delayed-order flags are calculated in PostgreSQL.

The On-Time Delivery KPI uses the following thresholds:

- Green: 95% or higher.
- Orange: 90% to below 95%.
- Red: below 90%.

**RFM Segmentation**

RFM is calculated in Python using eligible purchase records and loaded into PostgreSQL.

- **Recency:** Days since the customer's latest eligible purchase.
- **Frequency:** Number of distinct eligible orders.
- **Monetary:** Total product value purchased, excluding freight.

The five segments are Champion, At risk, Newbies, Regulars, and Sleeping.

The final segmentation contains **94,983 customers**. The remaining 437 customers do not meet RFM eligibility criteria and are not assigned an artificial sixth segment.

The SQL Mart provides the `current_rfm_segment` field for Power BI reporting.

## 4. Functional Requirements

The report contains four main business pages.

**Strategic Dashboard — Executive Director**

1. **KPI Overview**
   - Four headline KPI cards.
   - Monthly performance trend.
   - Product-category comparison.
   - Customer-state performance table.
   - Interactive metric selection and date filtering.

2. **Geographic Sales Analysis**
   - Geographic distribution of orders.
   - Revenue, Orders, and AOV metric selection.
   - Region → State → City drill-down.
   - Product-category and regional performance analysis.
   - Map visualization using geographical coordinates from PostgreSQL.

3. **Customer Segmentation RFM**
   - Customer and revenue KPIs.
   - Comparison of RFM segment performance.
   - Customer distribution across five segments.
   - RFM customer behavior table.
   - Interactive selection between Total Revenue and Revenue per Customer.

**Operational Dashboard — Operations Manager**

4. **Delivery Performance**
   - Total Delayed Orders.
   - On-Time Delivery %.
   - Avg Delay Days.
   - Avg Delivery Days.
   - Delivery Time Distribution.
   - Delivery trends and state-level delay analysis.
   - Order Status and date filtering.

The report supports interactive navigation, slicers, dynamic titles, and explanatory tooltips.

Additional tooltip, methodology, and Order Details pages support the four main business pages.

## 5. Non-Functional & Technical Requirements

**Security — Dynamic Row-Level Security**

Dynamic RLS restricts access to customer and sales data based on the authenticated user's assigned role and state.

The security mapping is stored in PostgreSQL in `mart.dim_security`, rather than in a manually maintained Power BI table.

- Managers can access their assigned state.
- Managers assigned `ALL` can access all states.
- Directors assigned `ALL` can access all states.
- Users without an authorized mapping cannot access data protected by the customer RLS rule.

Power BI uses `USERPRINCIPALNAME()` to identify the current user. The RLS rule is applied to the customer dimension and propagates to the fact table through the model relationship.

**Data Model**

Power BI uses the PostgreSQL Mart Star Schema.

- `mart.fact_sales` stores order-item-level transactions and reporting measures.
- `mart.dim_customer` stores customer attributes, SCD2 history, geographical coordinates, and RFM information.
- `mart.dim_product` stores product information.
- `mart.dim_seller` stores seller identifiers.
- `mart.dim_date` provides the reporting calendar.
- `mart.dim_security` provides authorization mappings.

The security table is intentionally disconnected from the Star Schema and is referenced by the RLS expression.

**DAX and Performance**

- SQL calculates delivery durations, delays, and the current RFM segment.
- DAX calculates reporting aggregations, averages, percentages, dynamic metric selections, and visualization titles.
- `SUMX` is used for the Total Revenue measure.
- `REMOVEFILTERS` supports total-context comparisons while respecting RLS.
- Month-over-month calculations support sales trend analysis.

Unused measures, including previous-year and YoY calculations, were removed from the current semantic model.

**Date Management**

The model uses `mart.dim_date` as its calendar, with Auto date/time disabled.

The active relationship uses the order purchase date. Four additional date relationships remain inactive and require a final decision on whether to use them in reporting or remove them.

**Connectivity and Refresh**

PostgreSQL connection settings are managed through the `pServer` and `pDatabase` parameters rather than hard-coded connection values.

The intended refresh frequency is daily. In the local development environment, SQL loads and Power BI refreshes are performed manually.

**Performance and Usability**

The report aims to provide responsive interactions through a Star Schema, reusable measures, consistent formatting, readable labels, and a limited number of headline KPIs per page.

## 6. Known Limitations and Pending Improvements

- Four current customer records contain incorrect geographical coordinates originating from the Olist geolocation data. The source ETL correction is pending.
- The map uses latitude and longitude from PostgreSQL, but validation in Power BI Service is still pending.
- The KPI Overview state table displays Top 10 states and Others. The Others row follows value-based sorting rather than being fixed at the bottom.
- The Geographic Sales Analysis location chart retains the working Region → State → City drill-down; the Top 10 + Others grouping has not been implemented there.
- Four inactive date relationships remain to be used in dedicated calculations or removed after the final model review.

STEP 3 - Wireframing & Mockups

Low-fidelity wireframes were prepared before the final Power BI implementation for:

- KPI Overview
- Map / Drill-down
- RFM Segmentation
- Operations Dashboard

The wireframes defined the initial placement of KPI cards, filters, navigation, and main supporting visuals.

During implementation, the Map / Drill-down page evolved into **Geographic Sales Analysis**, while the Operations Dashboard became **Delivery Performance**. The original wireframes are retained as evidence of the initial design process.

The wireframe images are included separately in the Stage 3 deliverables.

STEP 4

Pattern 1 - Customer value is concentrated in the Regulars segment

The Regulars segment contains the largest number of customers and generates the highest total revenue. At the same time, Champions represent only a very small part of the customer base but generate much higher revenue per customer.

Hypothesis:
Most customers make only a limited number of repeat purchases, while a small group of highly engaged customers generates significantly more value per customer.

Business Action:
The company should focus retention and repeat-purchase campaigns on high-value Regulars and At-risk customers. Converting even a small proportion of these customers into Champions could increase customer lifetime value without relying only on new customer acquisition.


Pattern 2 - Delivery delays show visible peaks during periods of higher activity

Most deliveries are completed within the 6-10 day delivery-time bucket, while the overall delayed order rate is approximately 6.77%. The delayed-order trend also shows several visible peaks, particularly during late 2017 and early 2018.

Hypothesis:
Periods with higher order activity may put additional pressure on fulfillment and logistics capacity, which can contribute to an increase in delayed deliveries.

Business Action:
Operations should monitor delayed-order rate together with order volume and prepare additional logistics capacity before expected demand peaks. State-level monitoring can then be used to identify locations where operational intervention is required.

STEP 5

Three aggregated dashboard metrics were independently validated against raw SQL queries executed directly on the Mart layer:

- Total Revenue
- Total Orders
- Total Delayed Orders

The Power BI results matched the corresponding SQL query results.

Side-by-side Power BI vs SQL screenshots are stored separately as part of the Stage 3 validation evidence.